In [ ]:
#@title Cell 26.1 - Notebook overview
from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 26: Nested Pathogen-Out Selection and Validation of Model C

## Purpose

Notebook 26 will select and validate the Model C pathogen kernel using the
9,058 pathogens retained in Notebook 25 and their observed MIC measurements.

The Model C pathogen kernel is

\[
K_P^{(C)}(\rho)
=
\rho K_{\mathrm{seq}}
+(1-\rho)K_P^{(3B)},
\qquad \rho\in[0,1].
\]

Two sequence kernels will be considered: the base-weighted kernel and the
locus-weighted kernel.

## Outer and inner pathogen folds

An **outer fold** is the group of pathogens held out for final evaluation.
Their MIC values are not used to select any model setting.

The remaining pathogens form the outer training group. This group is divided
again into **inner folds**. The inner folds select the sequence-kernel type,
\(\rho\), pathogen dimension \(r_C\), and Ridge penalty \(\alpha\).

The selected settings are then fitted using the complete outer training group
and evaluated on the held-out outer-fold pathogens.

## Computationally practical nested selection

Selection is performed in two stages inside every outer training group:

1. select the sequence-kernel type and \(\rho\) using the optimised Model 3B
   dimension and Ridge penalty stored by Notebook 16B; and
2. for the selected kernel, select \(r_C\) and \(\alpha\).

The same second-stage procedure is applied to the \(\rho=0\) Model 3B baseline.
This provides a fair baseline on the same pathogens and outer folds.

To avoid decomposing 21 full \(9058\times9058\) matrices separately, each of
the three kernel components is first represented by its leading 512 spectral
coordinates. Every combined candidate is then reduced to its leading 256
coordinates. The retained information and sample reconstruction error are
reported explicitly.

## Notebook operations

Notebook 26 will:

1. load the Notebook 25 kernel components and optimised Notebook 16B MIC data;
2. retain MIC observations from the 9,058 Model C pathogens;
3. calculate restartable spectral coordinates for the kernel components;
4. construct 21 unique Model C coordinate candidates;
5. define five outer and three inner BioSample-grouped pathogen folds;
6. perform restartable nested model selection within each outer training group;
7. calculate held-out-pathogen performance for Model C and Model 3B;
8. select the final Model C settings using all 9,058 pathogens;
9. fit the final Model C reference model; and
10. save the validation and fitted-model outputs.

## Model boundary

Pathogen similarity information remains available for every pathogen. Only the
held-out pathogens' MIC measurements are excluded from model fitting and model
selection within each outer fold.

The antibiotic kernel and its 26-dimensional coordinates remain unchanged.
The 319 pathogens without assembly sequences are not included.

Notebook 25 and Model 3B files are treated as read-only inputs.

## Expected notebook length

Notebook 26 contains **12 cells**. Cell 26.9 processes one outer fold per run
and should be rerun until all five outer folds are complete.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the Notebook 26 settings."
)


In [ ]:
# =============================================================================
# Cell 26.2
# =============================================================================

#@title Cell 26.2 - Import packages and define notebook settings
# This cell imports the required packages, mounts Google Drive and defines the
# fold counts, candidate dimensions, Ridge penalties and restart settings.

import gc
import hashlib
import json
import math
import os
import shutil
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from google.colab import drive
from IPython.display import display
from scipy.sparse.linalg import eigsh
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold


drive.mount(
    "/content/drive",
    force_remount=False,
)


EXPECTED_MODEL_C_PATHOGENS = 9058
EXPECTED_ANTIBIOTICS = 26
ANTIBIOTIC_COORDINATES = 26

NUMBER_OF_OUTER_FOLDS = 5
NUMBER_OF_INNER_FOLDS = 3
NUMBER_OF_FINAL_SELECTION_FOLDS = 3

PATHOGEN_DIMENSION_CANDIDATES = [
    16,
    32,
    64,
    128,
    256,
]

RIDGE_ALPHA_CANDIDATES = [
    1.0,
    10.0,
    100.0,
    1000.0,
]

CANDIDATE_SPECTRAL_RANK = max(
    PATHOGEN_DIMENSION_CANDIDATES
)

COMPONENT_SPECTRAL_RANK = 512

RANDOM_SEED = 42
EIGEN_TOLERANCE = 1e-4
EIGEN_MAXIMUM_ITERATIONS = 5000
RIDGE_TOLERANCE = 1e-4
RIDGE_MAXIMUM_ITERATIONS = 2000

OUTER_FOLDS_PER_RUN = 1
ROW_BLOCK_SIZE = 256
EMBEDDING_VALIDATION_SAMPLE_SIZE = 256


MYDRIVE_DIRECTORY = Path(
    "/content/drive/MyDrive"
)

PROJECT_DIRECTORY = (
    MYDRIVE_DIRECTORY
    / "Model3_MIC_Project"
)

NOTEBOOK25_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook25"
)

NOTEBOOK26_DIRECTORY = (
    PROJECT_DIRECTORY
    / "notebook26"
)

NOTEBOOK26_RESULT_DIRECTORY = (
    NOTEBOOK26_DIRECTORY
    / "results"
)

SPECTRAL_CHECKPOINT_DIRECTORY = (
    NOTEBOOK26_DIRECTORY
    / "spectral_checkpoints"
)

CANDIDATE_EMBEDDING_DIRECTORY = (
    NOTEBOOK26_DIRECTORY
    / "candidate_embeddings"
)

OUTER_FOLD_CHECKPOINT_DIRECTORY = (
    NOTEBOOK26_DIRECTORY
    / "outer_fold_checkpoints"
)

WORK_DIRECTORY = Path(
    "/content/notebook26_work"
)

INPUT_DIRECTORY = (
    WORK_DIRECTORY
    / "inputs"
)

NOTEBOOK25_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "notebook25"
)

MODEL3B_INPUT_DIRECTORY = (
    INPUT_DIRECTORY
    / "model3b"
)

for directory in [
    PROJECT_DIRECTORY,
    NOTEBOOK26_DIRECTORY,
    NOTEBOOK26_RESULT_DIRECTORY,
    SPECTRAL_CHECKPOINT_DIRECTORY,
    CANDIDATE_EMBEDDING_DIRECTORY,
    OUTER_FOLD_CHECKPOINT_DIRECTORY,
    WORK_DIRECTORY,
    INPUT_DIRECTORY,
    NOTEBOOK25_INPUT_DIRECTORY,
    MODEL3B_INPUT_DIRECTORY,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


NOTEBOOK25_ARCHIVE_FILENAME = (
    "25_model_c_pathogen_kernel_candidate_components.zip"
)

MODEL3B_ARCHIVE_FILENAME = (
    "16B_optimised_full_gene_model3_interaction_outputs.zip"
)


def file_sha256(file_path):
    digest = hashlib.sha256()

    with open(
        file_path,
        "rb",
    ) as input_file:
        for block in iter(
            lambda: input_file.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


settings_summary = pd.DataFrame(
    [
        {
            "setting": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "setting": "Outer pathogen folds",
            "value": NUMBER_OF_OUTER_FOLDS,
        },
        {
            "setting": "Inner pathogen folds",
            "value": NUMBER_OF_INNER_FOLDS,
        },
        {
            "setting": "Pathogen dimensions",
            "value": str(
                PATHOGEN_DIMENSION_CANDIDATES
            ),
        },
        {
            "setting": "Ridge alpha values",
            "value": str(
                RIDGE_ALPHA_CANDIDATES
            ),
        },
        {
            "setting": "Component spectral coordinates",
            "value": COMPONENT_SPECTRAL_RANK,
        },
        {
            "setting": "Maximum candidate coordinates",
            "value": CANDIDATE_SPECTRAL_RANK,
        },
        {
            "setting": "Outer folds processed per run",
            "value": OUTER_FOLDS_PER_RUN,
        },
        {
            "setting": "Notebook 26 output directory",
            "value": str(NOTEBOOK26_DIRECTORY),
        },
    ]
)

display(settings_summary)

print(
    "Notebook 26 packages, directories and fixed settings "
    "were defined successfully."
)

print(
    "\nTransition: Cell 26.3 will locate, validate and "
    "extract the Notebook 25 and optimised Notebook 16B inputs."
)


In [ ]:
# =============================================================================
# Cell 26.3
# =============================================================================

#@title Cell 26.3 - Locate, validate and extract the required archives
# This cell locates the Notebook 25 kernel archive and optimised Notebook 16B
# model archive, validates them and extracts only the files required here.


def locate_archive(
    authoritative_filename,
    preferred_paths,
):
    authoritative_path = Path(
        authoritative_filename
    )

    candidate_paths = []

    for preferred_path in preferred_paths:
        preferred_path = Path(
            preferred_path
        )

        if preferred_path.exists():
            candidate_paths.append(
                preferred_path.resolve()
            )

    if not candidate_paths:
        filename_stem = authoritative_path.stem

        candidate_paths = [
            path.resolve()
            for path in PROJECT_DIRECTORY.rglob(
                f"{filename_stem}*.zip"
            )
            if path.name.startswith(
                filename_stem
            )
        ]

    candidate_paths = sorted(
        set(candidate_paths),
        key=lambda path: (
            len(path.parts),
            str(path),
        ),
    )

    if not candidate_paths:
        raise FileNotFoundError(
            f"{authoritative_filename} was not found. "
            f"Copy it into {PROJECT_DIRECTORY} and rerun "
            "Cell 26.3."
        )

    if len(candidate_paths) > 1:
        candidate_hashes = {
            file_sha256(path)
            for path in candidate_paths
        }

        if len(candidate_hashes) > 1:
            raise ValueError(
                f"Multiple different candidate archives were "
                f"found for {authoritative_filename}: "
                f"{candidate_paths}. Retain one authoritative "
                "copy and rerun Cell 26.3."
            )

    return candidate_paths[0]


def archive_member_for_basename(
    archive,
    required_basename,
):
    matches = [
        member
        for member in archive.namelist()
        if Path(member).name
        == required_basename
    ]

    if len(matches) != 1:
        raise ValueError(
            f"Expected one archive member named "
            f"{required_basename}, but found {matches}."
        )

    return matches[0]


def validate_and_extract_members(
    archive_path,
    required_basenames,
    destination_directory,
):
    extracted_paths = {}

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        damaged_member = archive.testzip()

        if damaged_member is not None:
            raise ValueError(
                f"{archive_path.name} contains a damaged "
                f"member: {damaged_member}"
            )

        for required_basename in required_basenames:
            archive_member = archive_member_for_basename(
                archive,
                required_basename,
            )

            output_path = (
                destination_directory
                / required_basename
            )

            partial_path = output_path.with_suffix(
                output_path.suffix + ".partial"
            )

            partial_path.unlink(
                missing_ok=True
            )

            with archive.open(
                archive_member,
                "r",
            ) as source_file:
                with open(
                    partial_path,
                    "wb",
                ) as destination_file:
                    shutil.copyfileobj(
                        source_file,
                        destination_file,
                        length=1024 * 1024,
                    )

            partial_path.replace(
                output_path
            )

            extracted_paths[
                required_basename
            ] = output_path

    return extracted_paths


notebook25_archive_path = locate_archive(
    NOTEBOOK25_ARCHIVE_FILENAME,
    [
        NOTEBOOK25_DIRECTORY
        / NOTEBOOK25_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / NOTEBOOK25_ARCHIVE_FILENAME,
    ],
)

model3b_archive_path = locate_archive(
    MODEL3B_ARCHIVE_FILENAME,
    [
        PROJECT_DIRECTORY
        / MODEL3B_ARCHIVE_FILENAME,
        PROJECT_DIRECTORY
        / "notebook16B"
        / MODEL3B_ARCHIVE_FILENAME,
        MYDRIVE_DIRECTORY
        / MODEL3B_ARCHIVE_FILENAME,
    ],
)


notebook25_required_files = [
    "25_model3b_pathogen_kernel_subset.npy",
    "25_sequence_kernel_base_weighted.npy",
    "25_sequence_kernel_locus_weighted.npy",
    "25_model_c_pathogen_order.csv",
    "25_rho_candidate_grid.csv",
    "25_model_c_kernel_family_configuration.json",
    "25_output_manifest.json",
]

model3b_required_files = [
    "16B_optimised_model3_aligned_interactions.csv",
    "16B_optimised_model3_kernel_embeddings.npz",
    "16B_optimised_model3_antibiotic_embedding_index.csv",
    "16B_optimised_model3_configuration.json",
]


notebook25_input_paths = validate_and_extract_members(
    notebook25_archive_path,
    notebook25_required_files,
    NOTEBOOK25_INPUT_DIRECTORY,
)

model3b_input_paths = validate_and_extract_members(
    model3b_archive_path,
    model3b_required_files,
    MODEL3B_INPUT_DIRECTORY,
)


input_summary = pd.DataFrame(
    [
        {
            "input": "Notebook 25 kernel components",
            "archive": notebook25_archive_path.name,
            "required_files": len(
                notebook25_required_files
            ),
            "validation_status": "passed",
        },
        {
            "input": "Optimised Notebook 16B model data",
            "archive": model3b_archive_path.name,
            "required_files": len(
                model3b_required_files
            ),
            "validation_status": "passed",
        },
    ]
)

display(input_summary)

print(f"Notebook 25 archive: {notebook25_archive_path}")
print(f"Notebook 16B archive: {model3b_archive_path}")

print(
    "\nTransition: Cell 26.4 will load and align the "
    "Model C MIC observations and antibiotic coordinates."
)


In [ ]:
# =============================================================================
# Cell 26.4
# =============================================================================

#@title Cell 26.4 - Load and align MIC observations and antibiotic coordinates
# This cell retains observed MICs from the 9,058 Model C pathogens and assigns
# the correct pathogen row and antibiotic-coordinate row to every observation.

interactions = pd.read_csv(
    model3b_input_paths[
        "16B_optimised_model3_aligned_interactions.csv"
    ]
)

model_c_pathogen_order = pd.read_csv(
    notebook25_input_paths[
        "25_model_c_pathogen_order.csv"
    ]
)

antibiotic_index = pd.read_csv(
    model3b_input_paths[
        "16B_optimised_model3_antibiotic_embedding_index.csv"
    ]
)

with np.load(
    model3b_input_paths[
        "16B_optimised_model3_kernel_embeddings.npz"
    ],
    allow_pickle=False,
) as embedding_archive:
    antibiotic_embedding = np.asarray(
        embedding_archive[
            "antibiotic_embedding"
        ],
        dtype=np.float32,
    )

with open(
    model3b_input_paths[
        "16B_optimised_model3_configuration.json"
    ],
    "r",
    encoding="utf-8",
) as configuration_file:
    model3b_configuration = json.load(
        configuration_file
    )

REFERENCE_PATHOGEN_DIMENSION = int(
    model3b_configuration[
        "selected_pathogen_embedding_dimensions"
    ]
)

REFERENCE_RIDGE_ALPHA = float(
    model3b_configuration[
        "selected_ridge_alpha"
    ]
)

if REFERENCE_PATHOGEN_DIMENSION not in (
    PATHOGEN_DIMENSION_CANDIDATES
):
    raise ValueError(
        "The optimised Notebook 16B pathogen dimension is "
        "not included in the Notebook 26 candidates."
    )

if REFERENCE_RIDGE_ALPHA not in (
    RIDGE_ALPHA_CANDIDATES
):
    raise ValueError(
        "The optimised Notebook 16B Ridge penalty is not "
        "included in the Notebook 26 candidates."
    )


required_interaction_columns = {
    "biosample",
    "antibiotic",
    "log2_mic",
}

required_order_columns = {
    "model_c_row_index",
    "biosample",
    "assembly_accession",
}

required_antibiotic_columns = {
    "kernel_row",
    "antibiotic",
}

if not required_interaction_columns.issubset(
    interactions.columns
):
    raise ValueError(
        "The Notebook 16B interaction table is missing "
        "required columns."
    )

if not required_order_columns.issubset(
    model_c_pathogen_order.columns
):
    raise ValueError(
        "The Notebook 25 pathogen order is missing "
        "required columns."
    )

if not required_antibiotic_columns.issubset(
    antibiotic_index.columns
):
    raise ValueError(
        "The antibiotic index is missing required columns."
    )

model_c_pathogen_order = (
    model_c_pathogen_order
    .sort_values("model_c_row_index")
    .reset_index(drop=True)
)

if len(model_c_pathogen_order) != (
    EXPECTED_MODEL_C_PATHOGENS
):
    raise ValueError(
        "The Model C pathogen order has the wrong "
        "number of rows."
    )

if not np.array_equal(
    model_c_pathogen_order[
        "model_c_row_index"
    ].to_numpy(dtype=np.int64),
    np.arange(
        EXPECTED_MODEL_C_PATHOGENS,
        dtype=np.int64,
    ),
):
    raise ValueError(
        "The Model C pathogen order is not complete "
        "and consecutive."
    )

antibiotic_index = (
    antibiotic_index
    .sort_values("kernel_row")
    .reset_index(drop=True)
)

if antibiotic_embedding.shape != (
    EXPECTED_ANTIBIOTICS,
    ANTIBIOTIC_COORDINATES,
):
    raise ValueError(
        "The antibiotic coordinate matrix has the wrong "
        f"dimensions: {antibiotic_embedding.shape}."
    )

if len(antibiotic_index) != EXPECTED_ANTIBIOTICS:
    raise ValueError(
        "The antibiotic index has the wrong number of rows."
    )


biosample_to_model_c_row = dict(
    zip(
        model_c_pathogen_order[
            "biosample"
        ].astype(str),
        model_c_pathogen_order[
            "model_c_row_index"
        ].astype(int),
    )
)

antibiotic_to_row = dict(
    zip(
        antibiotic_index[
            "antibiotic"
        ].astype(str),
        antibiotic_index[
            "kernel_row"
        ].astype(int),
    )
)

interactions = interactions[
    interactions[
        "biosample"
    ].astype(str).isin(
        biosample_to_model_c_row
    )
].copy()

interactions[
    "model_c_pathogen_row"
] = interactions[
    "biosample"
].astype(str).map(
    biosample_to_model_c_row
)

interactions[
    "antibiotic_coordinate_row"
] = interactions[
    "antibiotic"
].astype(str).map(
    antibiotic_to_row
)

if interactions[
    [
        "model_c_pathogen_row",
        "antibiotic_coordinate_row",
    ]
].isna().any().any():
    raise ValueError(
        "Some Model C MIC observations could not be "
        "aligned with the kernel rows."
    )

interactions[
    "model_c_pathogen_row"
] = interactions[
    "model_c_pathogen_row"
].astype(int)

interactions[
    "antibiotic_coordinate_row"
] = interactions[
    "antibiotic_coordinate_row"
].astype(int)

interactions[
    "log2_mic"
] = pd.to_numeric(
    interactions["log2_mic"],
    errors="raise",
)

if interactions.duplicated(
    [
        "biosample",
        "antibiotic",
    ]
).any():
    raise ValueError(
        "Duplicate BioSample-antibiotic MIC observations "
        "were detected."
    )

if interactions[
    "biosample"
].nunique() != EXPECTED_MODEL_C_PATHOGENS:
    raise ValueError(
        "Not every Model C pathogen has an observed MIC "
        "after cohort filtering."
    )

if interactions[
    "antibiotic"
].nunique() != EXPECTED_ANTIBIOTICS:
    raise ValueError(
        "The filtered Model C cohort does not contain all "
        "26 antibiotics."
    )

interactions = interactions.reset_index(
    drop=True
)

MODEL_C_INTERACTIONS = len(interactions)

pathogen_rows = interactions[
    "model_c_pathogen_row"
].to_numpy(dtype=np.int64)

antibiotic_rows = interactions[
    "antibiotic_coordinate_row"
].to_numpy(dtype=np.int64)

response_values = interactions[
    "log2_mic"
].to_numpy(dtype=np.float64)

biosample_groups = interactions[
    "biosample"
].astype(str).to_numpy()

ALIGNED_INTERACTION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_model_c_aligned_interactions.csv.gz"
)

interactions.to_csv(
    ALIGNED_INTERACTION_PATH,
    index=False,
    compression="gzip",
)


interaction_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": interactions[
                "biosample"
            ].nunique(),
        },
        {
            "metric": "Antibiotics",
            "value": interactions[
                "antibiotic"
            ].nunique(),
        },
        {
            "metric": "Observed MIC interactions",
            "value": MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Antibiotic coordinates",
            "value": ANTIBIOTIC_COORDINATES,
        },
        {
            "metric": "Notebook 16B reference pathogen coordinates",
            "value": REFERENCE_PATHOGEN_DIMENSION,
        },
        {
            "metric": "Notebook 16B reference Ridge alpha",
            "value": REFERENCE_RIDGE_ALPHA,
        },
    ]
)

display(interaction_summary)

print(f"Saved: {ALIGNED_INTERACTION_PATH}")

print(
    "\nTransition: Cell 26.5 will load and validate "
    "the three aligned pathogen-kernel components."
)


In [ ]:
# =============================================================================
# Cell 26.5
# =============================================================================

#@title Cell 26.5 - Load and validate the kernel components
# This cell verifies the checksums and dimensions of the Model 3B subset,
# base-weighted sequence kernel and locus-weighted sequence kernel.

with open(
    notebook25_input_paths[
        "25_output_manifest.json"
    ],
    "r",
    encoding="utf-8",
) as manifest_file:
    notebook25_manifest = json.load(
        manifest_file
    )

with open(
    notebook25_input_paths[
        "25_model_c_kernel_family_configuration.json"
    ],
    "r",
    encoding="utf-8",
) as configuration_file:
    notebook25_configuration = json.load(
        configuration_file
    )

rho_candidates = np.asarray(
    notebook25_configuration[
        "rho_candidates"
    ],
    dtype=np.float64,
)

if not np.array_equal(
    rho_candidates,
    np.round(
        np.linspace(0.0, 1.0, 11),
        2,
    ),
):
    raise ValueError(
        "The Notebook 25 rho grid is not the agreed "
        "0.0 to 1.0 grid in steps of 0.1."
    )

manifest_file_records = {
    record["file_name"]: record
    for record in notebook25_manifest["files"]
}

kernel_component_paths = {
    "model3b": notebook25_input_paths[
        "25_model3b_pathogen_kernel_subset.npy"
    ],
    "base": notebook25_input_paths[
        "25_sequence_kernel_base_weighted.npy"
    ],
    "locus": notebook25_input_paths[
        "25_sequence_kernel_locus_weighted.npy"
    ],
}

kernel_component_labels = {
    "model3b": "Model 3B subset",
    "base": "Base-weighted sequence kernel",
    "locus": "Locus-weighted sequence kernel",
}

kernel_components = {}
kernel_component_summary_rows = []

for component_id, component_path in (
    kernel_component_paths.items()
):
    expected_record = manifest_file_records.get(
        component_path.name
    )

    if expected_record is None:
        raise ValueError(
            f"The Notebook 25 manifest does not contain "
            f"{component_path.name}."
        )

    observed_sha256 = file_sha256(
        component_path
    )

    if observed_sha256 != expected_record["sha256"]:
        raise ValueError(
            f"The checksum does not match for "
            f"{component_path.name}."
        )

    kernel = np.load(
        component_path,
        mmap_mode="r",
    )

    expected_shape = (
        EXPECTED_MODEL_C_PATHOGENS,
        EXPECTED_MODEL_C_PATHOGENS,
    )

    if kernel.shape != expected_shape:
        raise ValueError(
            f"{component_path.name} has dimensions "
            f"{kernel.shape}; expected {expected_shape}."
        )

    diagonal_difference = float(
        np.max(
            np.abs(
                np.asarray(
                    np.diagonal(kernel),
                    dtype=np.float64,
                )
                - 1.0
            )
        )
    )

    if diagonal_difference > 1e-6:
        raise ValueError(
            f"{component_path.name} has an invalid diagonal."
        )

    kernel_components[
        component_id
    ] = kernel

    kernel_component_summary_rows.append(
        {
            "component_id": component_id,
            "kernel_component":
                kernel_component_labels[
                    component_id
                ],
            "rows": kernel.shape[0],
            "columns": kernel.shape[1],
            "maximum_diagonal_difference":
                diagonal_difference,
            "sha256": observed_sha256,
            "validation_status": "passed",
        }
    )

kernel_component_summary = pd.DataFrame(
    kernel_component_summary_rows
)

KERNEL_COMPONENT_INPUT_VALIDATION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_kernel_component_input_validation.csv"
)

kernel_component_summary.to_csv(
    KERNEL_COMPONENT_INPUT_VALIDATION_PATH,
    index=False,
)

display(kernel_component_summary)

print(
    f"Saved: {KERNEL_COMPONENT_INPUT_VALIDATION_PATH}"
)

print(
    "\nTransition: Cell 26.6 will calculate or reuse "
    "the leading spectral coordinates of all three components."
)


In [ ]:
# =============================================================================
# Cell 26.6
# =============================================================================

#@title Cell 26.6 - Calculate restartable component spectral coordinates
# This cell calculates the leading 512 eigenvalues and eigenvectors of each
# kernel component. One checkpoint is saved per component so completed work is
# reused if the Colab runtime disconnects.


def normalise_eigenvector_signs(
    eigenvectors,
):
    eigenvectors = np.asarray(
        eigenvectors,
        dtype=np.float64,
    ).copy()

    for coordinate_index in range(
        eigenvectors.shape[1]
    ):
        coordinate = eigenvectors[
            :,
            coordinate_index,
        ]

        pivot_index = int(
            np.argmax(
                np.abs(coordinate)
            )
        )

        if coordinate[pivot_index] < 0:
            eigenvectors[
                :,
                coordinate_index,
            ] *= -1.0

    return eigenvectors


def calculate_component_spectrum(
    component_id,
    kernel,
    kernel_sha256,
):
    checkpoint_path = (
        SPECTRAL_CHECKPOINT_DIRECTORY
        / f"26_{component_id}_rank_"
          f"{COMPONENT_SPECTRAL_RANK}_spectrum.npz"
    )

    metadata_path = checkpoint_path.with_suffix(
        ".json"
    )

    reuse_checkpoint = False

    if checkpoint_path.exists() and metadata_path.exists():
        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as metadata_file:
            metadata = json.load(
                metadata_file
            )

        reuse_checkpoint = (
            metadata.get("component_id")
            == component_id
            and metadata.get("kernel_sha256")
            == kernel_sha256
            and metadata.get("spectral_rank")
            == COMPONENT_SPECTRAL_RANK
            and metadata.get("pathogens")
            == EXPECTED_MODEL_C_PATHOGENS
            and metadata.get("checkpoint_sha256")
            == file_sha256(checkpoint_path)
        )

        if reuse_checkpoint:
            with np.load(
                checkpoint_path,
                allow_pickle=False,
            ) as checkpoint:
                reuse_checkpoint = (
                    checkpoint[
                        "eigenvalues"
                    ].shape
                    == (
                        COMPONENT_SPECTRAL_RANK,
                    )
                    and checkpoint[
                        "eigenvectors"
                    ].shape
                    == (
                        EXPECTED_MODEL_C_PATHOGENS,
                        COMPONENT_SPECTRAL_RANK,
                    )
                )

    if reuse_checkpoint:
        print(
            f"Reusing {component_id} spectral checkpoint."
        )

    else:
        print(
            f"Calculating {component_id} component spectrum..."
        )

        random_generator = np.random.default_rng(
            RANDOM_SEED
        )

        starting_vector = random_generator.normal(
            size=EXPECTED_MODEL_C_PATHOGENS
        )

        eigenvalues, eigenvectors = eigsh(
            kernel,
            k=COMPONENT_SPECTRAL_RANK,
            which="LA",
            v0=starting_vector,
            tol=EIGEN_TOLERANCE,
            maxiter=EIGEN_MAXIMUM_ITERATIONS,
        )

        descending_order = np.argsort(
            eigenvalues
        )[::-1]

        eigenvalues = np.asarray(
            eigenvalues[descending_order],
            dtype=np.float64,
        )

        eigenvectors = normalise_eigenvector_signs(
            eigenvectors[
                :,
                descending_order,
            ]
        )

        if eigenvalues.min() < -1e-5:
            raise ValueError(
                f"{component_id} produced a materially "
                "negative retained eigenvalue."
            )

        eigenvalues = np.clip(
            eigenvalues,
            0.0,
            None,
        )

        local_checkpoint_path = (
            WORK_DIRECTORY
            / checkpoint_path.name
        )

        np.savez_compressed(
            local_checkpoint_path,
            eigenvalues=eigenvalues.astype(
                np.float32
            ),
            eigenvectors=eigenvectors.astype(
                np.float32
            ),
        )

        partial_checkpoint_path = (
            checkpoint_path.with_suffix(
                ".npz.partial"
            )
        )

        partial_checkpoint_path.unlink(
            missing_ok=True
        )

        shutil.copy2(
            local_checkpoint_path,
            partial_checkpoint_path,
        )

        partial_checkpoint_path.replace(
            checkpoint_path
        )

        metadata = {
            "component_id": component_id,
            "kernel_sha256": kernel_sha256,
            "spectral_rank":
                COMPONENT_SPECTRAL_RANK,
            "pathogens":
                EXPECTED_MODEL_C_PATHOGENS,
            "eigen_tolerance":
                EIGEN_TOLERANCE,
            "checkpoint_sha256":
                file_sha256(checkpoint_path),
        }

        partial_metadata_path = (
            metadata_path.with_suffix(
                ".json.partial"
            )
        )

        with open(
            partial_metadata_path,
            "w",
            encoding="utf-8",
        ) as metadata_file:
            json.dump(
                metadata,
                metadata_file,
                indent=2,
            )

        partial_metadata_path.replace(
            metadata_path
        )

    with np.load(
        checkpoint_path,
        allow_pickle=False,
    ) as checkpoint:
        eigenvalues = np.asarray(
            checkpoint["eigenvalues"],
            dtype=np.float64,
        )

    return {
        "component_id": component_id,
        "checkpoint_path": checkpoint_path,
        "retained_eigenvalues":
            len(eigenvalues),
        "largest_eigenvalue": float(
            eigenvalues[0]
        ),
        "smallest_retained_eigenvalue": float(
            eigenvalues[-1]
        ),
        "retained_trace_percentage": float(
            100.0
            * eigenvalues.sum()
            / EXPECTED_MODEL_C_PATHOGENS
        ),
        "validation_status": "passed",
    }


component_sha256_lookup = {
    row.component_id: row.sha256
    for row in kernel_component_summary.itertuples(
        index=False
    )
}

component_spectral_rows = []

for component_id in [
    "model3b",
    "base",
    "locus",
]:
    component_spectral_rows.append(
        calculate_component_spectrum(
            component_id,
            kernel_components[component_id],
            component_sha256_lookup[component_id],
        )
    )

component_spectral_summary = pd.DataFrame(
    component_spectral_rows
)

COMPONENT_SPECTRAL_SUMMARY_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_component_spectral_summary.csv"
)

component_spectral_summary.to_csv(
    COMPONENT_SPECTRAL_SUMMARY_PATH,
    index=False,
)

display(
    component_spectral_summary.drop(
        columns=["checkpoint_path"]
    )
)

print(f"Saved: {COMPONENT_SPECTRAL_SUMMARY_PATH}")

print(
    "\nTransition: Cell 26.7 will use the three component "
    "spectra to construct 21 unique Model C coordinate candidates."
)


In [ ]:
# =============================================================================
# Cell 26.7
# =============================================================================

#@title Cell 26.7 - Construct the 21 unique Model C coordinate candidates
# This cell combines the component spectral coordinates for each sequence type
# and rho value. The rho-zero baseline is stored once because both sequence
# types are identical when rho equals zero.


def load_component_coordinates(
    component_id,
):
    checkpoint_path = Path(
        component_spectral_summary.loc[
            component_spectral_summary[
                "component_id"
            ]
            == component_id,
            "checkpoint_path",
        ].iloc[0]
    )

    with np.load(
        checkpoint_path,
        allow_pickle=False,
    ) as checkpoint:
        eigenvalues = np.asarray(
            checkpoint["eigenvalues"],
            dtype=np.float64,
        )

        eigenvectors = np.asarray(
            checkpoint["eigenvectors"],
            dtype=np.float64,
        )

    coordinates = (
        eigenvectors
        * np.sqrt(
            eigenvalues
        )[None, :]
    )

    return eigenvalues, coordinates


component_spectra = {}

for component_id in [
    "model3b",
    "base",
    "locus",
]:
    component_spectra[
        component_id
    ] = load_component_coordinates(
        component_id
    )


candidate_definitions = [
    {
        "candidate_id": "model3b_rho_0p0",
        "sequence_kernel": "model3b-only",
        "sequence_component_id": "none",
        "rho": 0.0,
    }
]

for sequence_kernel, component_id in [
    ("base-weighted", "base"),
    ("locus-weighted", "locus"),
]:
    for rho in rho_candidates[
        rho_candidates > 0
    ]:
        rho_text = (
            f"{rho:.1f}"
            .replace(".", "p")
        )

        candidate_definitions.append(
            {
                "candidate_id": (
                    f"{component_id}_rho_{rho_text}"
                ),
                "sequence_kernel":
                    sequence_kernel,
                "sequence_component_id":
                    component_id,
                "rho": float(rho),
            }
        )

if len(candidate_definitions) != 21:
    raise AssertionError(
        "Expected 21 unique Model C kernel candidates."
    )


sample_indices = np.linspace(
    0,
    EXPECTED_MODEL_C_PATHOGENS - 1,
    EMBEDDING_VALIDATION_SAMPLE_SIZE,
    dtype=int,
)

candidate_catalog_rows = []

model3b_eigenvalues, model3b_coordinates = (
    component_spectra["model3b"]
)

for candidate_number, candidate in enumerate(
    candidate_definitions,
    start=1,
):
    candidate_id = candidate[
        "candidate_id"
    ]

    rho = float(candidate["rho"])

    checkpoint_path = (
        CANDIDATE_EMBEDDING_DIRECTORY
        / f"26_{candidate_id}_embedding.npz"
    )

    metadata_path = checkpoint_path.with_suffix(
        ".json"
    )

    reuse_checkpoint = False

    if checkpoint_path.exists() and metadata_path.exists():
        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as metadata_file:
            metadata = json.load(
                metadata_file
            )

        reuse_checkpoint = (
            metadata.get("candidate_id")
            == candidate_id
            and metadata.get("rho")
            == rho
            and metadata.get("maximum_dimension")
            == CANDIDATE_SPECTRAL_RANK
            and metadata.get("component_spectral_rank")
            == COMPONENT_SPECTRAL_RANK
            and metadata.get("pathogens")
            == EXPECTED_MODEL_C_PATHOGENS
            and metadata.get("checkpoint_sha256")
            == file_sha256(checkpoint_path)
        )

        if reuse_checkpoint:
            try:
                with np.load(
                    checkpoint_path,
                    allow_pickle=False,
                ) as checkpoint:
                    reuse_checkpoint = (
                        checkpoint[
                            "pathogen_embedding"
                        ].shape
                        == (
                            EXPECTED_MODEL_C_PATHOGENS,
                            CANDIDATE_SPECTRAL_RANK,
                        )
                        and checkpoint[
                            "pathogen_eigenvalues"
                        ].shape
                        == (
                            CANDIDATE_SPECTRAL_RANK,
                        )
                    )
            except Exception:
                reuse_checkpoint = False

    if reuse_checkpoint:
        print(
            f"[{candidate_number}/21] Reusing {candidate_id}."
        )

        with open(
            metadata_path,
            "r",
            encoding="utf-8",
        ) as metadata_file:
            metadata = json.load(
                metadata_file
            )

        candidate_catalog_rows.append(
            metadata
        )

        continue

    if rho == 0.0:
        candidate_eigenvalues = (
            model3b_eigenvalues[
                :CANDIDATE_SPECTRAL_RANK
            ].copy()
        )

        candidate_embedding = (
            model3b_coordinates[
                :,
                :CANDIDATE_SPECTRAL_RANK,
            ].copy()
        )

        sequence_component_id = "none"

    else:
        sequence_component_id = candidate[
            "sequence_component_id"
        ]

        sequence_eigenvalues, sequence_coordinates = (
            component_spectra[
                sequence_component_id
            ]
        )

        combined_factor = np.concatenate(
            [
                np.sqrt(1.0 - rho)
                * model3b_coordinates,
                np.sqrt(rho)
                * sequence_coordinates,
            ],
            axis=1,
        )

        gram_matrix = np.asarray(
            combined_factor.T
            @ combined_factor,
            dtype=np.float64,
        )

        gram_eigenvalues, gram_eigenvectors = (
            np.linalg.eigh(
                gram_matrix
            )
        )

        descending_order = np.argsort(
            gram_eigenvalues
        )[::-1][
            :CANDIDATE_SPECTRAL_RANK
        ]

        candidate_eigenvalues = np.clip(
            gram_eigenvalues[
                descending_order
            ],
            0.0,
            None,
        )

        candidate_embedding = (
            combined_factor
            @ gram_eigenvectors[
                :,
                descending_order,
            ]
        )

        candidate_embedding = (
            normalise_eigenvector_signs(
                candidate_embedding
            )
        )

        del combined_factor
        del gram_matrix
        gc.collect()

    candidate_embedding = np.asarray(
        candidate_embedding,
        dtype=np.float32,
    )

    candidate_eigenvalues = np.asarray(
        candidate_eigenvalues,
        dtype=np.float32,
    )

    if candidate_embedding.shape != (
        EXPECTED_MODEL_C_PATHOGENS,
        CANDIDATE_SPECTRAL_RANK,
    ):
        raise ValueError(
            f"{candidate_id} has the wrong embedding "
            f"dimensions: {candidate_embedding.shape}."
        )

    local_checkpoint_path = (
        WORK_DIRECTORY
        / checkpoint_path.name
    )

    np.savez_compressed(
        local_checkpoint_path,
        pathogen_embedding=
            candidate_embedding,
        pathogen_eigenvalues=
            candidate_eigenvalues,
    )

    partial_checkpoint_path = (
        checkpoint_path.with_suffix(
            ".npz.partial"
        )
    )

    partial_checkpoint_path.unlink(
        missing_ok=True
    )

    shutil.copy2(
        local_checkpoint_path,
        partial_checkpoint_path,
    )

    partial_checkpoint_path.replace(
        checkpoint_path
    )

    if rho == 0.0:
        actual_sample = np.asarray(
            kernel_components["model3b"][
                np.ix_(
                    sample_indices,
                    sample_indices,
                )
            ],
            dtype=np.float64,
        )
    else:
        actual_sample = (
            (1.0 - rho)
            * np.asarray(
                kernel_components["model3b"][
                    np.ix_(
                        sample_indices,
                        sample_indices,
                    )
                ],
                dtype=np.float64,
            )
            + rho
            * np.asarray(
                kernel_components[
                    sequence_component_id
                ][
                    np.ix_(
                        sample_indices,
                        sample_indices,
                    )
                ],
                dtype=np.float64,
            )
        )

    approximate_sample = (
        candidate_embedding[
            sample_indices,
            :,
        ].astype(np.float64)
        @ candidate_embedding[
            sample_indices,
            :,
        ].astype(np.float64).T
    )

    approximation_difference = (
        actual_sample
        - approximate_sample
    )

    metadata = {
        "candidate_id": candidate_id,
        "sequence_kernel": candidate[
            "sequence_kernel"
        ],
        "sequence_component_id":
            sequence_component_id,
        "rho": rho,
        "model3b_contribution": 1.0 - rho,
        "sequence_contribution": rho,
        "pathogens":
            EXPECTED_MODEL_C_PATHOGENS,
        "maximum_dimension":
            CANDIDATE_SPECTRAL_RANK,
        "component_spectral_rank":
            COMPONENT_SPECTRAL_RANK,
        "embedding_file":
            checkpoint_path.name,
        "retained_trace_percentage": float(
            100.0
            * candidate_eigenvalues.sum()
            / EXPECTED_MODEL_C_PATHOGENS
        ),
        "sample_approximation_rmse": float(
            np.sqrt(
                np.mean(
                    np.square(
                        approximation_difference
                    )
                )
            )
        ),
        "sample_maximum_absolute_difference": float(
            np.max(
                np.abs(
                    approximation_difference
                )
            )
        ),
        "checkpoint_sha256":
            file_sha256(checkpoint_path),
        "validation_status": "passed",
    }

    partial_metadata_path = (
        metadata_path.with_suffix(
            ".json.partial"
        )
    )

    with open(
        partial_metadata_path,
        "w",
        encoding="utf-8",
    ) as metadata_file:
        json.dump(
            metadata,
            metadata_file,
            indent=2,
        )

    partial_metadata_path.replace(
        metadata_path
    )

    candidate_catalog_rows.append(
        metadata
    )

    print(
        f"[{candidate_number}/21] Saved {candidate_id}: "
        f"rho={rho:.1f}, "
        f"sample RMSE="
        f"{metadata['sample_approximation_rmse']:.4f}"
    )


candidate_catalog = pd.DataFrame(
    candidate_catalog_rows
).sort_values(
    [
        "rho",
        "sequence_kernel",
    ]
).reset_index(drop=True)

if len(candidate_catalog) != 21:
    raise AssertionError(
        "The candidate catalog does not contain "
        "21 unique entries."
    )

if candidate_catalog[
    "candidate_id"
].duplicated().any():
    raise AssertionError(
        "Duplicate candidate identifiers were detected."
    )

CANDIDATE_CATALOG_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_model_c_candidate_embedding_catalog.csv"
)

candidate_catalog.to_csv(
    CANDIDATE_CATALOG_PATH,
    index=False,
)

display(
    candidate_catalog[
        [
            "candidate_id",
            "sequence_kernel",
            "rho",
            "retained_trace_percentage",
            "sample_approximation_rmse",
            "validation_status",
        ]
    ]
)

print(f"Saved: {CANDIDATE_CATALOG_PATH}")

print(
    "\nTransition: Cell 26.8 will define and validate "
    "the five outer BioSample-grouped pathogen folds."
)


In [ ]:
# =============================================================================
# Cell 26.8
# =============================================================================

#@title Cell 26.8 - Define and validate the outer pathogen folds
# This cell assigns every BioSample to one of five outer folds and confirms
# that no BioSample contributes MIC observations to both training and
# evaluation within the same fold.

outer_grouped_folds = GroupKFold(
    n_splits=NUMBER_OF_OUTER_FOLDS
)

outer_splits = list(
    outer_grouped_folds.split(
        np.zeros(
            MODEL_C_INTERACTIONS,
            dtype=np.uint8,
        ),
        response_values,
        groups=biosample_groups,
    )
)

interaction_outer_fold = np.full(
    MODEL_C_INTERACTIONS,
    -1,
    dtype=np.int16,
)

outer_fold_summary_rows = []

for fold_number, (
    training_indices,
    evaluation_indices,
) in enumerate(
    outer_splits,
    start=1,
):
    training_biosamples = set(
        biosample_groups[
            training_indices
        ]
    )

    evaluation_biosamples = set(
        biosample_groups[
            evaluation_indices
        ]
    )

    overlap = (
        training_biosamples
        & evaluation_biosamples
    )

    if overlap:
        raise ValueError(
            f"Outer fold {fold_number} contains "
            "BioSample leakage."
        )

    interaction_outer_fold[
        evaluation_indices
    ] = fold_number

    outer_fold_summary_rows.append(
        {
            "outer_fold": fold_number,
            "training_biosamples":
                len(training_biosamples),
            "evaluation_biosamples":
                len(evaluation_biosamples),
            "training_interactions":
                len(training_indices),
            "evaluation_interactions":
                len(evaluation_indices),
            "biosample_overlap":
                len(overlap),
            "validation_status": "passed",
        }
    )

if (interaction_outer_fold < 1).any():
    raise AssertionError(
        "At least one interaction was not assigned to an "
        "outer fold."
    )

outer_fold_summary = pd.DataFrame(
    outer_fold_summary_rows
)

biosample_fold_assignment = (
    interactions[
        [
            "biosample",
        ]
    ]
    .assign(
        outer_fold=interaction_outer_fold
    )
    .drop_duplicates()
    .sort_values("biosample")
    .reset_index(drop=True)
)

if biosample_fold_assignment[
    "biosample"
].duplicated().any():
    raise AssertionError(
        "A BioSample was assigned to multiple outer folds."
    )

OUTER_FOLD_SUMMARY_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_outer_fold_summary.csv"
)

OUTER_FOLD_ASSIGNMENT_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_biosample_outer_fold_assignments.csv"
)

outer_fold_summary.to_csv(
    OUTER_FOLD_SUMMARY_PATH,
    index=False,
)

biosample_fold_assignment.to_csv(
    OUTER_FOLD_ASSIGNMENT_PATH,
    index=False,
)

display(outer_fold_summary)

print(f"Saved: {OUTER_FOLD_SUMMARY_PATH}")
print(f"Saved: {OUTER_FOLD_ASSIGNMENT_PATH}")

print(
    "\nTransition: Cell 26.9 will perform restartable "
    "nested selection and evaluation, one outer fold per run."
)


In [ ]:
# =============================================================================
# Cell 26.9
# =============================================================================

#@title Cell 26.9 - Run restartable nested pathogen-out selection
# This cell selects settings entirely within each outer training group and then
# evaluates the selected Model C model on the held-out outer-fold pathogens. It
# also tunes and evaluates the rho-zero Model 3B baseline on the same folds.
# One pending outer fold is processed per run.


def safe_pearson_correlation(
    actual_values,
    predicted_values,
):
    actual_values = np.asarray(
        actual_values,
        dtype=np.float64,
    )

    predicted_values = np.asarray(
        predicted_values,
        dtype=np.float64,
    )

    if (
        len(actual_values) < 2
        or np.std(actual_values) == 0
        or np.std(predicted_values) == 0
    ):
        return np.nan

    return float(
        np.corrcoef(
            actual_values,
            predicted_values,
        )[0, 1]
    )


def load_candidate_embedding(
    candidate_id,
):
    candidate_row = candidate_catalog.loc[
        candidate_catalog[
            "candidate_id"
        ]
        == candidate_id
    ]

    if len(candidate_row) != 1:
        raise ValueError(
            f"Candidate {candidate_id} was not found "
            "exactly once."
        )

    embedding_file = (
        CANDIDATE_EMBEDDING_DIRECTORY
        / candidate_row[
            "embedding_file"
        ].iloc[0]
    )

    if not embedding_file.exists():
        raise FileNotFoundError(
            f"The embedding file for {candidate_id} is missing."
        )

    if file_sha256(
        embedding_file
    ) != candidate_row[
        "checkpoint_sha256"
    ].iloc[0]:
        raise ValueError(
            f"The embedding checksum failed for {candidate_id}."
        )

    with np.load(
        embedding_file,
        allow_pickle=False,
    ) as embedding_archive:
        embedding = np.asarray(
            embedding_archive[
                "pathogen_embedding"
            ],
            dtype=np.float32,
        )

        eigenvalues = np.asarray(
            embedding_archive[
                "pathogen_eigenvalues"
            ],
            dtype=np.float32,
        )

    if embedding.shape != (
        EXPECTED_MODEL_C_PATHOGENS,
        CANDIDATE_SPECTRAL_RANK,
    ):
        raise ValueError(
            f"The embedding dimensions are invalid for {candidate_id}."
        )

    if eigenvalues.shape != (
        CANDIDATE_SPECTRAL_RANK,
    ):
        raise ValueError(
            f"The eigenvalue dimensions are invalid for {candidate_id}."
        )

    return (
        candidate_row.iloc[0].to_dict(),
        embedding,
        eigenvalues,
    )


def construct_interaction_design(
    pathogen_embedding,
    interaction_indices,
    pathogen_dimension,
):
    interaction_indices = np.asarray(
        interaction_indices,
        dtype=np.int64,
    )

    pathogen_coordinates = pathogen_embedding[
        pathogen_rows[interaction_indices],
        :pathogen_dimension,
    ].astype(
        np.float32,
        copy=False,
    )

    antibiotic_coordinates = antibiotic_embedding[
        antibiotic_rows[interaction_indices],
        :,
    ].astype(
        np.float32,
        copy=False,
    )

    design_matrix = np.einsum(
        "nr,ns->nrs",
        pathogen_coordinates,
        antibiotic_coordinates,
        optimize=True,
    ).reshape(
        len(interaction_indices),
        pathogen_dimension
        * ANTIBIOTIC_COORDINATES,
    )

    return np.asarray(
        design_matrix,
        dtype=np.float32,
    )


def fit_ridge_and_predict(
    training_design,
    training_response,
    evaluation_design,
    alpha,
):
    model = Ridge(
        alpha=float(alpha),
        fit_intercept=True,
        solver="lsqr",
        tol=RIDGE_TOLERANCE,
        max_iter=RIDGE_MAXIMUM_ITERATIONS,
        copy_X=False,
    )

    model.fit(
        training_design,
        training_response,
    )

    predictions = model.predict(
        evaluation_design
    )

    return model, np.asarray(
        predictions,
        dtype=np.float64,
    )


def evaluate_inner_grid(
    candidate_metadata,
    pathogen_embedding,
    outer_training_indices,
    inner_splits,
    pathogen_dimensions,
    alpha_values,
    outer_fold_number,
    selection_stage,
):
    maximum_dimension = max(
        pathogen_dimensions
    )

    maximum_design = construct_interaction_design(
        pathogen_embedding,
        outer_training_indices,
        maximum_dimension,
    )

    outer_training_response = response_values[
        outer_training_indices
    ]

    result_rows = []

    for pathogen_dimension in pathogen_dimensions:
        interaction_features = (
            pathogen_dimension
            * ANTIBIOTIC_COORDINATES
        )

        dimension_design = maximum_design[
            :,
            :interaction_features,
        ]

        for alpha in alpha_values:
            inner_mae_values = []
            inner_rmse_values = []

            for (
                inner_training_relative,
                inner_evaluation_relative,
            ) in inner_splits:
                training_design = np.asarray(
                    dimension_design[
                        inner_training_relative,
                        :,
                    ],
                    dtype=np.float32,
                )

                evaluation_design = np.asarray(
                    dimension_design[
                        inner_evaluation_relative,
                        :,
                    ],
                    dtype=np.float32,
                )

                training_response = (
                    outer_training_response[
                        inner_training_relative
                    ]
                )

                evaluation_response = (
                    outer_training_response[
                        inner_evaluation_relative
                    ]
                )

                _, inner_predictions = (
                    fit_ridge_and_predict(
                        training_design,
                        training_response,
                        evaluation_design,
                        alpha,
                    )
                )

                inner_mae_values.append(
                    mean_absolute_error(
                        evaluation_response,
                        inner_predictions,
                    )
                )

                inner_rmse_values.append(
                    math.sqrt(
                        mean_squared_error(
                            evaluation_response,
                            inner_predictions,
                        )
                    )
                )

                del training_design
                del evaluation_design
                gc.collect()

            rmse_standard_deviation = float(
                np.std(
                    inner_rmse_values,
                    ddof=1,
                )
            )

            rmse_standard_error = float(
                rmse_standard_deviation
                / math.sqrt(
                    len(inner_rmse_values)
                )
            )

            result_rows.append(
                {
                    "outer_fold":
                        outer_fold_number,
                    "selection_stage":
                        selection_stage,
                    "candidate_id":
                        candidate_metadata[
                            "candidate_id"
                        ],
                    "sequence_kernel":
                        candidate_metadata[
                            "sequence_kernel"
                        ],
                    "rho": float(
                        candidate_metadata[
                            "rho"
                        ]
                    ),
                    "pathogen_coordinates":
                        pathogen_dimension,
                    "antibiotic_coordinates":
                        ANTIBIOTIC_COORDINATES,
                    "interaction_features":
                        interaction_features,
                    "alpha": float(alpha),
                    "mean_inner_mae": float(
                        np.mean(inner_mae_values)
                    ),
                    "mean_inner_rmse": float(
                        np.mean(inner_rmse_values)
                    ),
                    "rmse_standard_deviation":
                        rmse_standard_deviation,
                    "rmse_standard_error":
                        rmse_standard_error,
                    "selected": False,
                }
            )

    del maximum_design
    gc.collect()

    return pd.DataFrame(
        result_rows
    )


def select_kernel_candidate(
    selection_results,
):
    ordered_results = selection_results.sort_values(
        [
            "mean_inner_rmse",
            "rho",
            "sequence_kernel",
            "candidate_id",
        ],
        ascending=[
            True,
            True,
            True,
            True,
        ],
    )

    return ordered_results.iloc[0]


def select_dimension_and_alpha(
    selection_results,
):
    best_result = selection_results.sort_values(
        [
            "mean_inner_rmse",
            "pathogen_coordinates",
            "alpha",
        ]
    ).iloc[0]

    threshold = float(
        best_result["mean_inner_rmse"]
        + best_result["rmse_standard_error"]
    )

    near_best_results = selection_results[
        selection_results[
            "mean_inner_rmse"
        ]
        <= threshold + 1e-12
    ]

    selected_dimension = int(
        near_best_results[
            "pathogen_coordinates"
        ].min()
    )

    selected_result = (
        near_best_results[
            near_best_results[
                "pathogen_coordinates"
            ]
            == selected_dimension
        ]
        .sort_values(
            [
                "mean_inner_rmse",
                "alpha",
            ]
        )
        .iloc[0]
    )

    return (
        selected_result,
        float(best_result["mean_inner_rmse"]),
        threshold,
    )


def fit_outer_model(
    pathogen_embedding,
    pathogen_dimension,
    alpha,
    training_indices,
    evaluation_indices,
):
    training_design = construct_interaction_design(
        pathogen_embedding,
        training_indices,
        pathogen_dimension,
    )

    evaluation_design = construct_interaction_design(
        pathogen_embedding,
        evaluation_indices,
        pathogen_dimension,
    )

    model, predictions = fit_ridge_and_predict(
        training_design,
        response_values[training_indices],
        evaluation_design,
        alpha,
    )

    del training_design
    del evaluation_design
    gc.collect()

    return model, predictions


def write_csv_atomically(
    table,
    output_path,
    compression=None,
):
    partial_path = Path(
        str(output_path) + ".partial"
    )

    partial_path.unlink(
        missing_ok=True
    )

    table.to_csv(
        partial_path,
        index=False,
        compression=compression,
    )

    partial_path.replace(
        output_path
    )


def outer_fold_checkpoint_paths(
    fold_number,
):
    prefix = f"26_outer_fold_{fold_number:02d}"

    return {
        "predictions": (
            OUTER_FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_predictions.csv.gz"
        ),
        "inner_results": (
            OUTER_FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_inner_selection.csv"
        ),
        "fold_summary": (
            OUTER_FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_summary.csv"
        ),
        "completion": (
            OUTER_FOLD_CHECKPOINT_DIRECTORY
            / f"{prefix}_complete.json"
        ),
    }


def outer_fold_checkpoint_is_valid(
    fold_number,
):
    paths = outer_fold_checkpoint_paths(
        fold_number
    )

    if not paths["completion"].exists():
        return False

    try:
        with open(
            paths["completion"],
            "r",
            encoding="utf-8",
        ) as completion_file:
            completion = json.load(
                completion_file
            )

        for file_key in [
            "predictions",
            "inner_results",
            "fold_summary",
        ]:
            file_path = paths[file_key]

            if (
                not file_path.exists()
                or file_sha256(file_path)
                != completion[
                    "file_sha256"
                ][file_key]
            ):
                return False

        return (
            completion.get("outer_fold")
            == fold_number
            and completion.get(
                "validation_status"
            )
            == "passed"
        )

    except Exception:
        return False


completed_outer_folds = [
    fold_number
    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
    if outer_fold_checkpoint_is_valid(
        fold_number
    )
]

pending_outer_folds = [
    fold_number
    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
    if fold_number
    not in completed_outer_folds
]

folds_to_process = pending_outer_folds[
    :OUTER_FOLDS_PER_RUN
]

print(
    f"Completed outer folds: {completed_outer_folds}"
)

print(
    f"Pending outer folds: {pending_outer_folds}"
)

print(
    f"This run will process: {folds_to_process}"
)


baseline_candidate_id = "model3b_rho_0p0"

for outer_fold_number in folds_to_process:
    print(
        f"\nStarting outer fold {outer_fold_number}."
    )

    outer_training_indices, outer_evaluation_indices = (
        outer_splits[
            outer_fold_number - 1
        ]
    )

    outer_training_groups = biosample_groups[
        outer_training_indices
    ]

    inner_grouped_folds = GroupKFold(
        n_splits=NUMBER_OF_INNER_FOLDS
    )

    inner_splits = list(
        inner_grouped_folds.split(
            np.zeros(
                len(outer_training_indices),
                dtype=np.uint8,
            ),
            response_values[
                outer_training_indices
            ],
            groups=outer_training_groups,
        )
    )

    for (
        inner_training_relative,
        inner_evaluation_relative,
    ) in inner_splits:
        inner_training_groups = set(
            outer_training_groups[
                inner_training_relative
            ]
        )

        inner_evaluation_groups = set(
            outer_training_groups[
                inner_evaluation_relative
            ]
        )

        if (
            inner_training_groups
            & inner_evaluation_groups
        ):
            raise ValueError(
                f"Outer fold {outer_fold_number} contains "
                "inner-fold BioSample leakage."
            )

    kernel_selection_tables = []

    for candidate_number, candidate_row in enumerate(
        candidate_catalog.itertuples(
            index=False
        ),
        start=1,
    ):
        candidate_metadata, candidate_embedding, _ = (
            load_candidate_embedding(
                candidate_row.candidate_id
            )
        )

        candidate_result = evaluate_inner_grid(
            candidate_metadata,
            candidate_embedding,
            outer_training_indices,
            inner_splits,
            [REFERENCE_PATHOGEN_DIMENSION],
            [REFERENCE_RIDGE_ALPHA],
            outer_fold_number,
            "sequence_kernel_and_rho",
        )

        kernel_selection_tables.append(
            candidate_result
        )

        print(
            f"Outer fold {outer_fold_number}, "
            f"kernel candidate {candidate_number}/21: "
            f"{candidate_row.candidate_id}, "
            f"RMSE="
            f"{candidate_result['mean_inner_rmse'].iloc[0]:.4f}"
        )

        del candidate_embedding
        gc.collect()

    kernel_selection_results = pd.concat(
        kernel_selection_tables,
        ignore_index=True,
    )

    selected_kernel_result = select_kernel_candidate(
        kernel_selection_results
    )

    selected_candidate_id = selected_kernel_result[
        "candidate_id"
    ]

    kernel_selection_results.loc[
        kernel_selection_results[
            "candidate_id"
        ]
        == selected_candidate_id,
        "selected",
    ] = True

    (
        selected_candidate_metadata,
        selected_candidate_embedding,
        _,
    ) = load_candidate_embedding(
        selected_candidate_id
    )

    selected_model_grid = evaluate_inner_grid(
        selected_candidate_metadata,
        selected_candidate_embedding,
        outer_training_indices,
        inner_splits,
        PATHOGEN_DIMENSION_CANDIDATES,
        RIDGE_ALPHA_CANDIDATES,
        outer_fold_number,
        "selected_model_dimension_and_alpha",
    )

    (
        selected_model_result,
        selected_best_inner_rmse,
        selected_near_best_threshold,
    ) = select_dimension_and_alpha(
        selected_model_grid
    )

    selected_model_grid.loc[
        (
            selected_model_grid[
                "pathogen_coordinates"
            ]
            == int(
                selected_model_result[
                    "pathogen_coordinates"
                ]
            )
        )
        & (
            selected_model_grid["alpha"]
            == float(
                selected_model_result["alpha"]
            )
        ),
        "selected",
    ] = True

    if selected_candidate_id == baseline_candidate_id:
        baseline_metadata = (
            selected_candidate_metadata
        )

        baseline_embedding = (
            selected_candidate_embedding.copy()
        )

        baseline_model_grid = (
            selected_model_grid.copy()
        )

        baseline_model_grid[
            "selection_stage"
        ] = "baseline_dimension_and_alpha"

    else:
        (
            baseline_metadata,
            baseline_embedding,
            _,
        ) = load_candidate_embedding(
            baseline_candidate_id
        )

        baseline_model_grid = evaluate_inner_grid(
            baseline_metadata,
            baseline_embedding,
            outer_training_indices,
            inner_splits,
            PATHOGEN_DIMENSION_CANDIDATES,
            RIDGE_ALPHA_CANDIDATES,
            outer_fold_number,
            "baseline_dimension_and_alpha",
        )

    (
        baseline_model_result,
        baseline_best_inner_rmse,
        baseline_near_best_threshold,
    ) = select_dimension_and_alpha(
        baseline_model_grid
    )

    baseline_model_grid["selected"] = False

    baseline_model_grid.loc[
        (
            baseline_model_grid[
                "pathogen_coordinates"
            ]
            == int(
                baseline_model_result[
                    "pathogen_coordinates"
                ]
            )
        )
        & (
            baseline_model_grid["alpha"]
            == float(
                baseline_model_result["alpha"]
            )
        ),
        "selected",
    ] = True

    selected_dimension = int(
        selected_model_result[
            "pathogen_coordinates"
        ]
    )

    selected_alpha = float(
        selected_model_result["alpha"]
    )

    baseline_dimension = int(
        baseline_model_result[
            "pathogen_coordinates"
        ]
    )

    baseline_alpha = float(
        baseline_model_result["alpha"]
    )

    _, selected_predictions = fit_outer_model(
        selected_candidate_embedding,
        selected_dimension,
        selected_alpha,
        outer_training_indices,
        outer_evaluation_indices,
    )

    _, baseline_predictions = fit_outer_model(
        baseline_embedding,
        baseline_dimension,
        baseline_alpha,
        outer_training_indices,
        outer_evaluation_indices,
    )

    evaluation_actual = response_values[
        outer_evaluation_indices
    ]

    selected_mae = mean_absolute_error(
        evaluation_actual,
        selected_predictions,
    )

    selected_rmse = math.sqrt(
        mean_squared_error(
            evaluation_actual,
            selected_predictions,
        )
    )

    selected_pearson = safe_pearson_correlation(
        evaluation_actual,
        selected_predictions,
    )

    baseline_mae = mean_absolute_error(
        evaluation_actual,
        baseline_predictions,
    )

    baseline_rmse = math.sqrt(
        mean_squared_error(
            evaluation_actual,
            baseline_predictions,
        )
    )

    baseline_pearson = safe_pearson_correlation(
        evaluation_actual,
        baseline_predictions,
    )

    prediction_base = interactions.loc[
        outer_evaluation_indices,
        [
            "biosample",
            "antibiotic",
            "log2_mic",
            "model_c_pathogen_row",
            "antibiotic_coordinate_row",
        ],
    ].copy()

    prediction_base.insert(
        0,
        "interaction_row",
        np.asarray(
            outer_evaluation_indices,
            dtype=np.int64,
        ),
    )

    prediction_base = prediction_base.rename(
        columns={
            "log2_mic": "observed_log2_mic"
        }
    )

    selected_prediction_table = (
        prediction_base.copy()
    )

    selected_prediction_table[
        "model"
    ] = "Model C"

    selected_prediction_table[
        "predicted_log2_mic"
    ] = selected_predictions

    selected_prediction_table[
        "outer_fold"
    ] = outer_fold_number

    selected_prediction_table[
        "candidate_id"
    ] = selected_candidate_id

    selected_prediction_table[
        "sequence_kernel"
    ] = selected_candidate_metadata[
        "sequence_kernel"
    ]

    selected_prediction_table["rho"] = float(
        selected_candidate_metadata["rho"]
    )

    selected_prediction_table[
        "pathogen_coordinates"
    ] = selected_dimension

    selected_prediction_table[
        "alpha"
    ] = selected_alpha

    baseline_prediction_table = prediction_base.copy()

    baseline_prediction_table[
        "model"
    ] = "Model 3B baseline"

    baseline_prediction_table[
        "predicted_log2_mic"
    ] = baseline_predictions

    baseline_prediction_table[
        "outer_fold"
    ] = outer_fold_number

    baseline_prediction_table[
        "candidate_id"
    ] = baseline_candidate_id

    baseline_prediction_table[
        "sequence_kernel"
    ] = "model3b-only"

    baseline_prediction_table["rho"] = 0.0

    baseline_prediction_table[
        "pathogen_coordinates"
    ] = baseline_dimension

    baseline_prediction_table[
        "alpha"
    ] = baseline_alpha

    fold_predictions = pd.concat(
        [
            selected_prediction_table,
            baseline_prediction_table,
        ],
        ignore_index=True,
    )

    fold_inner_results = pd.concat(
        [
            kernel_selection_results,
            selected_model_grid,
            baseline_model_grid,
        ],
        ignore_index=True,
    )

    fold_summary = pd.DataFrame(
        [
            {
                "outer_fold": outer_fold_number,
                "training_biosamples": len(
                    set(
                        biosample_groups[
                            outer_training_indices
                        ]
                    )
                ),
                "evaluation_biosamples": len(
                    set(
                        biosample_groups[
                            outer_evaluation_indices
                        ]
                    )
                ),
                "training_interactions":
                    len(outer_training_indices),
                "evaluation_interactions":
                    len(outer_evaluation_indices),
                "selected_candidate_id":
                    selected_candidate_id,
                "selected_sequence_kernel":
                    selected_candidate_metadata[
                        "sequence_kernel"
                    ],
                "selected_rho": float(
                    selected_candidate_metadata[
                        "rho"
                    ]
                ),
                "selected_pathogen_dimension":
                    selected_dimension,
                "selected_ridge_alpha": selected_alpha,
                "selected_best_inner_rmse":
                    selected_best_inner_rmse,
                "selected_near_best_threshold":
                    selected_near_best_threshold,
                "model_c_mae": selected_mae,
                "model_c_rmse": selected_rmse,
                "model_c_pearson_r":
                    selected_pearson,
                "baseline_pathogen_dimension":
                    baseline_dimension,
                "baseline_ridge_alpha": baseline_alpha,
                "baseline_best_inner_rmse":
                    baseline_best_inner_rmse,
                "baseline_near_best_threshold":
                    baseline_near_best_threshold,
                "baseline_mae": baseline_mae,
                "baseline_rmse": baseline_rmse,
                "baseline_pearson_r":
                    baseline_pearson,
                "biosample_overlap": 0,
                "validation_status": "passed",
            }
        ]
    )

    checkpoint_paths = outer_fold_checkpoint_paths(
        outer_fold_number
    )

    write_csv_atomically(
        fold_predictions,
        checkpoint_paths["predictions"],
        compression="gzip",
    )

    write_csv_atomically(
        fold_inner_results,
        checkpoint_paths["inner_results"],
    )

    write_csv_atomically(
        fold_summary,
        checkpoint_paths["fold_summary"],
    )

    completion = {
        "outer_fold": outer_fold_number,
        "file_sha256": {
            "predictions": file_sha256(
                checkpoint_paths["predictions"]
            ),
            "inner_results": file_sha256(
                checkpoint_paths["inner_results"]
            ),
            "fold_summary": file_sha256(
                checkpoint_paths["fold_summary"]
            ),
        },
        "validation_status": "passed",
    }

    partial_completion_path = Path(
        str(checkpoint_paths["completion"])
        + ".partial"
    )

    with open(
        partial_completion_path,
        "w",
        encoding="utf-8",
    ) as completion_file:
        json.dump(
            completion,
            completion_file,
            indent=2,
        )

    partial_completion_path.replace(
        checkpoint_paths["completion"]
    )

    print(
        f"Outer fold {outer_fold_number} completed: "
        f"selected {selected_candidate_id}, "
        f"r={selected_dimension}, "
        f"alpha={selected_alpha:g}, "
        f"Model C RMSE={selected_rmse:.4f}, "
        f"baseline RMSE={baseline_rmse:.4f}."
    )

    del selected_candidate_embedding
    del baseline_embedding
    gc.collect()


completed_after_run = [
    fold_number
    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
    if outer_fold_checkpoint_is_valid(
        fold_number
    )
]

print(
    f"\nCompleted outer folds after this run: "
    f"{completed_after_run}"
)

if len(completed_after_run) < NUMBER_OF_OUTER_FOLDS:
    print(
        "Rerun Cell 26.9 to process the next pending "
        "outer fold. Do not run Cell 26.10 yet."
    )
else:
    print(
        "All five outer folds are complete. "
        "Cell 26.10 can now combine the validation results."
    )


In [ ]:
# =============================================================================
# Cell 26.10
# =============================================================================

#@title Cell 26.10 - Combine and compare the nested pathogen-out results
# This cell confirms that all five outer folds are complete, combines their
# held-out predictions and reports Model C performance beside the rho=0
# Model 3B baseline on exactly the same MIC observations.

required_outer_fold_files = []

for outer_fold_number in range(
    1,
    NUMBER_OF_OUTER_FOLDS + 1,
):
    if not outer_fold_checkpoint_is_valid(
        outer_fold_number
    ):
        required_outer_fold_files.append(
            outer_fold_number
        )

if required_outer_fold_files:
    raise FileNotFoundError(
        "Outer-fold checkpoints are incomplete for folds "
        f"{required_outer_fold_files}. Rerun Cell 26.9 until "
        "all five folds are complete."
    )


prediction_tables = []
inner_result_tables = []
outer_summary_tables = []

for outer_fold_number in range(
    1,
    NUMBER_OF_OUTER_FOLDS + 1,
):
    checkpoint_paths = outer_fold_checkpoint_paths(
        outer_fold_number
    )

    prediction_tables.append(
        pd.read_csv(
            checkpoint_paths["predictions"]
        )
    )

    inner_result_tables.append(
        pd.read_csv(
            checkpoint_paths["inner_results"]
        )
    )

    outer_summary_tables.append(
        pd.read_csv(
            checkpoint_paths["fold_summary"]
        )
    )


nested_predictions = pd.concat(
    prediction_tables,
    ignore_index=True,
)

nested_inner_results = pd.concat(
    inner_result_tables,
    ignore_index=True,
)

nested_outer_results = pd.concat(
    outer_summary_tables,
    ignore_index=True,
)

expected_prediction_rows = (
    2 * MODEL_C_INTERACTIONS
)

if len(nested_predictions) != expected_prediction_rows:
    raise ValueError(
        "The combined prediction table has the wrong number "
        f"of rows: expected {expected_prediction_rows:,}, "
        f"observed {len(nested_predictions):,}."
    )

prediction_identity_columns = [
    "model",
    "interaction_row",
]

if nested_predictions.duplicated(
    prediction_identity_columns
).any():
    raise ValueError(
        "At least one MIC observation has more than one "
        "held-out prediction for the same model."
    )

model_counts = (
    nested_predictions[
        "model"
    ].value_counts()
)

for model_name in [
    "Model C",
    "Model 3B baseline",
]:
    if int(model_counts.get(model_name, 0)) != MODEL_C_INTERACTIONS:
        raise ValueError(
            f"{model_name} does not contain one held-out "
            "prediction for every Model C MIC observation."
        )


overall_performance_rows = []

for model_name, model_table in nested_predictions.groupby(
    "model",
    sort=False,
):
    observed = model_table[
        "observed_log2_mic"
    ].to_numpy(dtype=np.float64)

    predicted = model_table[
        "predicted_log2_mic"
    ].to_numpy(dtype=np.float64)

    overall_performance_rows.append(
        {
            "model": model_name,
            "observations": len(model_table),
            "pathogens": model_table[
                "biosample"
            ].nunique(),
            "mae": mean_absolute_error(
                observed,
                predicted,
            ),
            "rmse": math.sqrt(
                mean_squared_error(
                    observed,
                    predicted,
                )
            ),
            "pearson_r": safe_pearson_correlation(
                observed,
                predicted,
            ),
        }
    )

overall_performance = pd.DataFrame(
    overall_performance_rows
)

performance_lookup = overall_performance.set_index(
    "model"
)

model_c_performance = performance_lookup.loc[
    "Model C"
]

baseline_performance = performance_lookup.loc[
    "Model 3B baseline"
]

model_comparison = pd.DataFrame(
    [
        {
            "metric": "MAE improvement",
            "calculation": "Model 3B baseline minus Model C",
            "value": float(
                baseline_performance["mae"]
                - model_c_performance["mae"]
            ),
            "positive_value_favours": "Model C",
        },
        {
            "metric": "RMSE improvement",
            "calculation": "Model 3B baseline minus Model C",
            "value": float(
                baseline_performance["rmse"]
                - model_c_performance["rmse"]
            ),
            "positive_value_favours": "Model C",
        },
        {
            "metric": "Pearson correlation improvement",
            "calculation": "Model C minus Model 3B baseline",
            "value": float(
                model_c_performance["pearson_r"]
                - baseline_performance["pearson_r"]
            ),
            "positive_value_favours": "Model C",
        },
    ]
)


antibiotic_performance_rows = []

for (
    model_name,
    antibiotic_name,
), model_antibiotic_table in nested_predictions.groupby(
    [
        "model",
        "antibiotic",
    ],
    sort=True,
):
    observed = model_antibiotic_table[
        "observed_log2_mic"
    ].to_numpy(dtype=np.float64)

    predicted = model_antibiotic_table[
        "predicted_log2_mic"
    ].to_numpy(dtype=np.float64)

    antibiotic_performance_rows.append(
        {
            "model": model_name,
            "antibiotic": antibiotic_name,
            "observations": len(
                model_antibiotic_table
            ),
            "pathogens": model_antibiotic_table[
                "biosample"
            ].nunique(),
            "mae": mean_absolute_error(
                observed,
                predicted,
            ),
            "rmse": math.sqrt(
                mean_squared_error(
                    observed,
                    predicted,
                )
            ),
            "pearson_r": safe_pearson_correlation(
                observed,
                predicted,
            ),
        }
    )

antibiotic_performance = pd.DataFrame(
    antibiotic_performance_rows
)


selection_frequency = (
    nested_outer_results[
        [
            "selected_candidate_id",
            "selected_sequence_kernel",
            "selected_rho",
            "selected_pathogen_dimension",
            "selected_ridge_alpha",
        ]
    ]
    .value_counts(
        dropna=False
    )
    .rename("outer_folds")
    .reset_index()
    .sort_values(
        [
            "outer_folds",
            "selected_candidate_id",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


NESTED_PREDICTION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_nested_pathogen_out_predictions.csv.gz"
)

NESTED_INNER_RESULT_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_nested_inner_selection_results.csv.gz"
)

NESTED_OUTER_RESULT_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_nested_outer_fold_results.csv"
)

OVERALL_PERFORMANCE_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_nested_overall_performance.csv"
)

MODEL_COMPARISON_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_model_c_vs_model3b_comparison.csv"
)

ANTIBIOTIC_PERFORMANCE_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_nested_antibiotic_performance.csv"
)

SELECTION_FREQUENCY_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_outer_selection_frequency.csv"
)

nested_predictions.to_csv(
    NESTED_PREDICTION_PATH,
    index=False,
    compression="gzip",
)

nested_inner_results.to_csv(
    NESTED_INNER_RESULT_PATH,
    index=False,
    compression="gzip",
)

nested_outer_results.to_csv(
    NESTED_OUTER_RESULT_PATH,
    index=False,
)

overall_performance.to_csv(
    OVERALL_PERFORMANCE_PATH,
    index=False,
)

model_comparison.to_csv(
    MODEL_COMPARISON_PATH,
    index=False,
)

antibiotic_performance.to_csv(
    ANTIBIOTIC_PERFORMANCE_PATH,
    index=False,
)

selection_frequency.to_csv(
    SELECTION_FREQUENCY_PATH,
    index=False,
)

display(overall_performance)
display(model_comparison)
display(selection_frequency)

print(f"Saved: {NESTED_PREDICTION_PATH}")
print(f"Saved: {NESTED_INNER_RESULT_PATH}")
print(f"Saved: {NESTED_OUTER_RESULT_PATH}")
print(f"Saved: {OVERALL_PERFORMANCE_PATH}")
print(f"Saved: {MODEL_COMPARISON_PATH}")
print(f"Saved: {ANTIBIOTIC_PERFORMANCE_PATH}")
print(f"Saved: {SELECTION_FREQUENCY_PATH}")

print(
    "\nTransition: Cell 26.11 will select the final "
    "Model C settings using all Model C MIC observations "
    "and fit the final reference model."
)


In [ ]:
# =============================================================================
# Cell 26.11
# =============================================================================

#@title Cell 26.11 - Select and fit the final Model C reference model
# This cell repeats the two-stage selection using all 9,058 Model C pathogens,
# then fits one final reference model. Its training-fit values are reported only
# as a technical check; the independent performance estimate is from Cell 26.10.

FULL_COHORT_KERNEL_SELECTION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_full_cohort_kernel_selection.csv"
)

FULL_COHORT_MODEL_SELECTION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_full_cohort_dimension_alpha_selection.csv"
)

FINAL_SELECTION_SUMMARY_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_selection_summary.csv"
)

FINAL_MODEL_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_reference_model.joblib"
)

FINAL_EMBEDDING_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_embeddings.npz"
)

FINAL_PATHOGEN_INDEX_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_pathogen_embedding_index.csv"
)

FINAL_ANTIBIOTIC_INDEX_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_antibiotic_embedding_index.csv"
)

FINAL_CONFIGURATION_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_configuration.json"
)

FINAL_TRAINING_CHECK_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_final_model_c_training_fit_check.csv"
)


all_interaction_indices = np.arange(
    MODEL_C_INTERACTIONS,
    dtype=np.int64,
)

final_selection_splitter = GroupKFold(
    n_splits=NUMBER_OF_FINAL_SELECTION_FOLDS
)

final_selection_splits = list(
    final_selection_splitter.split(
        all_interaction_indices,
        response_values,
        groups=biosample_groups,
    )
)

for selection_train_indices, selection_test_indices in final_selection_splits:
    if set(
        biosample_groups[
            selection_train_indices
        ]
    ).intersection(
        set(
            biosample_groups[
                selection_test_indices
            ]
        )
    ):
        raise ValueError(
            "A BioSample occurs in both sides of a final-selection fold."
        )


def selected_value_mask(
    selected_values,
):
    if pd.api.types.is_bool_dtype(
        selected_values
    ):
        return selected_values

    return (
        selected_values
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("true")
    )


if FULL_COHORT_KERNEL_SELECTION_PATH.exists():
    full_cohort_kernel_selection = pd.read_csv(
        FULL_COHORT_KERNEL_SELECTION_PATH
    )

    expected_candidate_ids = set(
        candidate_catalog[
            "candidate_id"
        ]
    )

    observed_candidate_ids = set(
        full_cohort_kernel_selection[
            "candidate_id"
        ]
    )

    if observed_candidate_ids != expected_candidate_ids:
        raise ValueError(
            "The saved full-cohort kernel-selection table "
            "does not contain the current candidate set."
        )

    print(
        "Reusing the saved full-cohort kernel-selection results."
    )
else:
    full_cohort_kernel_selection_tables = []

    for candidate_number, candidate_row in enumerate(
        candidate_catalog.itertuples(index=False),
        start=1,
    ):
        print(
            f"Final kernel selection "
            f"[{candidate_number}/{len(candidate_catalog)}]: "
            f"{candidate_row.candidate_id}"
        )

        (
            candidate_metadata,
            candidate_embedding,
            _,
        ) = load_candidate_embedding(
            candidate_row.candidate_id
        )

        candidate_result = evaluate_inner_grid(
            candidate_metadata,
            candidate_embedding,
            all_interaction_indices,
            final_selection_splits,
            [
                REFERENCE_PATHOGEN_DIMENSION
            ],
            [
                REFERENCE_RIDGE_ALPHA
            ],
            0,
            "full_cohort_kernel_selection",
        )

        candidate_result[
            "sequence_kernel"
        ] = candidate_row.sequence_kernel

        candidate_result[
            "rho"
        ] = candidate_row.rho

        full_cohort_kernel_selection_tables.append(
            candidate_result
        )

        del candidate_embedding
        gc.collect()

    full_cohort_kernel_selection = pd.concat(
        full_cohort_kernel_selection_tables,
        ignore_index=True,
    )

    full_cohort_kernel_selection[
        "selected"
    ] = False

    selected_full_cohort_kernel = select_kernel_candidate(
        full_cohort_kernel_selection
    )

    full_cohort_kernel_selection.loc[
        full_cohort_kernel_selection[
            "candidate_id"
        ]
        == selected_full_cohort_kernel[
            "candidate_id"
        ],
        "selected",
    ] = True

    full_cohort_kernel_selection.to_csv(
        FULL_COHORT_KERNEL_SELECTION_PATH,
        index=False,
    )


selected_kernel_rows = full_cohort_kernel_selection[
    selected_value_mask(
        full_cohort_kernel_selection[
            "selected"
        ]
    )
]

if len(selected_kernel_rows) != 1:
    raise ValueError(
        "Exactly one full-cohort kernel candidate must be selected."
    )

selected_kernel_row = selected_kernel_rows.iloc[0]

FINAL_CANDIDATE_ID = str(
    selected_kernel_row[
        "candidate_id"
    ]
)

FINAL_SEQUENCE_KERNEL = str(
    selected_kernel_row[
        "sequence_kernel"
    ]
)

FINAL_RHO = float(
    selected_kernel_row[
        "rho"
    ]
)


(
    final_candidate_metadata,
    final_candidate_embedding,
    final_candidate_eigenvalues,
) = load_candidate_embedding(
    FINAL_CANDIDATE_ID
)

if FULL_COHORT_MODEL_SELECTION_PATH.exists():
    full_cohort_model_selection = pd.read_csv(
        FULL_COHORT_MODEL_SELECTION_PATH
    )

    if set(
        full_cohort_model_selection[
            "candidate_id"
        ]
    ) != {FINAL_CANDIDATE_ID}:
        raise ValueError(
            "The saved dimension-and-alpha selection belongs "
            "to a different kernel candidate."
        )

    print(
        "Reusing the saved full-cohort dimension-and-alpha results."
    )
else:
    full_cohort_model_selection = evaluate_inner_grid(
        final_candidate_metadata,
        final_candidate_embedding,
        all_interaction_indices,
        final_selection_splits,
        PATHOGEN_DIMENSION_CANDIDATES,
        RIDGE_ALPHA_CANDIDATES,
        0,
        "full_cohort_dimension_alpha_selection",
    )

    full_cohort_model_selection[
        "sequence_kernel"
    ] = FINAL_SEQUENCE_KERNEL

    full_cohort_model_selection[
        "rho"
    ] = FINAL_RHO

    (
        selected_full_cohort_model,
        full_cohort_best_inner_rmse,
        full_cohort_near_best_threshold,
    ) = select_dimension_and_alpha(
        full_cohort_model_selection
    )

    full_cohort_model_selection.loc[
        (
            full_cohort_model_selection[
                "pathogen_coordinates"
            ]
            == int(
                selected_full_cohort_model[
                    "pathogen_coordinates"
                ]
            )
        )
        & (
            full_cohort_model_selection[
                "alpha"
            ]
            == float(
                selected_full_cohort_model[
                    "alpha"
                ]
            )
        ),
        "selected",
    ] = True

    full_cohort_model_selection.to_csv(
        FULL_COHORT_MODEL_SELECTION_PATH,
        index=False,
    )


selected_model_rows = full_cohort_model_selection[
    selected_value_mask(
        full_cohort_model_selection[
            "selected"
        ]
    )
]

if len(selected_model_rows) != 1:
    raise ValueError(
        "Exactly one pathogen dimension and Ridge penalty "
        "must be selected."
    )

selected_model_row = selected_model_rows.iloc[0]

FINAL_PATHOGEN_DIMENSION = int(
    selected_model_row[
        "pathogen_coordinates"
    ]
)

FINAL_RIDGE_ALPHA = float(
    selected_model_row[
        "alpha"
    ]
)

FINAL_INTERACTION_DIMENSION = int(
    FINAL_PATHOGEN_DIMENSION
    * ANTIBIOTIC_COORDINATES
)


final_design = construct_interaction_design(
    final_candidate_embedding,
    all_interaction_indices,
    FINAL_PATHOGEN_DIMENSION,
)

final_model = Ridge(
    alpha=FINAL_RIDGE_ALPHA,
    fit_intercept=True,
    solver="lsqr",
    tol=RIDGE_TOLERANCE,
    max_iter=RIDGE_MAXIMUM_ITERATIONS,
    copy_X=False,
)

final_model.fit(
    final_design,
    response_values,
)

final_training_predictions = final_model.predict(
    final_design
)

if (
    not np.isfinite(final_model.coef_).all()
    or not np.isfinite(final_model.intercept_)
    or not np.isfinite(final_training_predictions).all()
):
    raise ValueError(
        "The fitted final Model C model contains invalid values."
    )

final_training_check = pd.DataFrame(
    [
        {
            "metric": "Training observations",
            "value": MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Training MAE",
            "value": mean_absolute_error(
                response_values,
                final_training_predictions,
            ),
        },
        {
            "metric": "Training RMSE",
            "value": math.sqrt(
                mean_squared_error(
                    response_values,
                    final_training_predictions,
                )
            ),
        },
        {
            "metric": "Training Pearson correlation",
            "value": safe_pearson_correlation(
                response_values,
                final_training_predictions,
            ),
        },
        {
            "metric": "Interpretation",
            "value": (
                "Technical fit check only; use nested outer-fold "
                "results for independent performance"
            ),
        },
    ]
)

del final_design
gc.collect()


joblib.dump(
    final_model,
    FINAL_MODEL_PATH,
    compress=3,
)

final_candidate_eigenvalues = np.asarray(
    final_candidate_eigenvalues,
    dtype=np.float64,
)[
    :FINAL_PATHOGEN_DIMENSION
]

np.savez_compressed(
    FINAL_EMBEDDING_PATH,
    pathogen_embedding=np.asarray(
        final_candidate_embedding[
            :,
            :FINAL_PATHOGEN_DIMENSION,
        ],
        dtype=np.float32,
    ),
    pathogen_eigenvalues=np.asarray(
        final_candidate_eigenvalues,
        dtype=np.float64,
    ),
    antibiotic_embedding=np.asarray(
        antibiotic_embedding,
        dtype=np.float32,
    ),
)


final_pathogen_index_columns = [
    column_name
    for column_name in [
        "model_c_row_index",
        "model_3b_row_index",
        "biosample",
        "assembly_accession",
    ]
    if column_name in model_c_pathogen_order.columns
]

final_pathogen_index = model_c_pathogen_order[
    final_pathogen_index_columns
].copy()

final_pathogen_index.insert(
    0,
    "pathogen_embedding_row",
    np.arange(
        EXPECTED_MODEL_C_PATHOGENS,
        dtype=np.int64,
    ),
)

final_pathogen_index.to_csv(
    FINAL_PATHOGEN_INDEX_PATH,
    index=False,
)

final_antibiotic_index = antibiotic_index.copy()

if "antibiotic_embedding_row" not in final_antibiotic_index.columns:
    final_antibiotic_index.insert(
        0,
        "antibiotic_embedding_row",
        np.arange(
            len(final_antibiotic_index),
            dtype=np.int64,
        ),
    )

final_antibiotic_index.to_csv(
    FINAL_ANTIBIOTIC_INDEX_PATH,
    index=False,
)


final_selection_summary = pd.DataFrame(
    [
        {
            "setting": "Sequence-kernel candidate",
            "value": FINAL_SEQUENCE_KERNEL,
        },
        {
            "setting": "Sequence contribution rho",
            "value": FINAL_RHO,
        },
        {
            "setting": "Pathogen coordinates r_C",
            "value": FINAL_PATHOGEN_DIMENSION,
        },
        {
            "setting": "Ridge alpha",
            "value": FINAL_RIDGE_ALPHA,
        },
        {
            "setting": "Antibiotic coordinates",
            "value": ANTIBIOTIC_COORDINATES,
        },
        {
            "setting": "Interaction predictors",
            "value": FINAL_INTERACTION_DIMENSION,
        },
    ]
)

final_selection_summary.to_csv(
    FINAL_SELECTION_SUMMARY_PATH,
    index=False,
)

final_training_check.to_csv(
    FINAL_TRAINING_CHECK_PATH,
    index=False,
)


final_configuration = {
    "notebook": 26,
    "model": "Model C",
    "target": "observed log2 MIC",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "observed_interactions": MODEL_C_INTERACTIONS,
    "antibiotic_coordinates": ANTIBIOTIC_COORDINATES,
    "kernel_formula": (
        "K_P_C = rho*K_seq + (1-rho)*K_P_3B"
    ),
    "selected_candidate_id": FINAL_CANDIDATE_ID,
    "selected_sequence_kernel": FINAL_SEQUENCE_KERNEL,
    "selected_rho": FINAL_RHO,
    "selected_pathogen_dimension": FINAL_PATHOGEN_DIMENSION,
    "selected_ridge_alpha": FINAL_RIDGE_ALPHA,
    "interaction_predictors": FINAL_INTERACTION_DIMENSION,
    "interaction_feature_order": (
        "For pathogen coordinate 1, all 26 antibiotic "
        "coordinates are listed first; this is repeated for "
        "each remaining pathogen coordinate."
    ),
    "candidate_rho_values": rho_candidates.tolist(),
    "candidate_pathogen_dimensions": (
        PATHOGEN_DIMENSION_CANDIDATES
    ),
    "candidate_ridge_alphas": RIDGE_ALPHA_CANDIDATES,
    "outer_validation_folds": NUMBER_OF_OUTER_FOLDS,
    "inner_selection_folds": NUMBER_OF_INNER_FOLDS,
    "final_selection_folds": NUMBER_OF_FINAL_SELECTION_FOLDS,
    "ridge_solver": "lsqr",
    "selection_method": {
        "stage_1": (
            "select sequence-kernel type and rho using "
            f"r={REFERENCE_PATHOGEN_DIMENSION} and "
            f"alpha={REFERENCE_RIDGE_ALPHA}"
        ),
        "stage_2": (
            "select pathogen dimension by the one-standard-error "
            "rule and select the lowest mean-RMSE alpha within it"
        ),
    },
    "performance_interpretation": (
        "Use the nested outer-fold results for independent "
        "performance. The final training-fit values are not an "
        "independent validation result."
    ),
    "validation_status": "passed",
}

temporary_configuration_path = Path(
    str(FINAL_CONFIGURATION_PATH)
    + ".partial"
)

with open(
    temporary_configuration_path,
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        final_configuration,
        configuration_file,
        indent=2,
    )

temporary_configuration_path.replace(
    FINAL_CONFIGURATION_PATH
)

display(final_selection_summary)
display(final_training_check)

print(f"Saved: {FULL_COHORT_KERNEL_SELECTION_PATH}")
print(f"Saved: {FULL_COHORT_MODEL_SELECTION_PATH}")
print(f"Saved: {FINAL_SELECTION_SUMMARY_PATH}")
print(f"Saved: {FINAL_MODEL_PATH}")
print(f"Saved: {FINAL_EMBEDDING_PATH}")
print(f"Saved: {FINAL_PATHOGEN_INDEX_PATH}")
print(f"Saved: {FINAL_ANTIBIOTIC_INDEX_PATH}")
print(f"Saved: {FINAL_CONFIGURATION_PATH}")
print(f"Saved: {FINAL_TRAINING_CHECK_PATH}")

print(
    "\nTransition: Cell 26.12 will package the validated "
    "Notebook 26 results and final Model C reference model."
)


In [ ]:
# =============================================================================
# Cell 26.12
# =============================================================================

#@title Cell 26.12 - Package and report the final Notebook 26 outputs
# This cell records file checksums, creates one validated ZIP archive and
# reports the final nested-validation and fitted-model status.

FINAL_OUTPUT_ARCHIVE_PATH = (
    NOTEBOOK26_DIRECTORY
    / "26_model_c_nested_pathogen_out_and_reference_model_outputs.zip"
)

OUTPUT_MANIFEST_PATH = (
    NOTEBOOK26_RESULT_DIRECTORY
    / "26_output_manifest.json"
)

files_to_package = [
    ALIGNED_INTERACTION_PATH,
    KERNEL_COMPONENT_INPUT_VALIDATION_PATH,
    COMPONENT_SPECTRAL_SUMMARY_PATH,
    CANDIDATE_CATALOG_PATH,
    OUTER_FOLD_SUMMARY_PATH,
    OUTER_FOLD_ASSIGNMENT_PATH,
    NESTED_PREDICTION_PATH,
    NESTED_INNER_RESULT_PATH,
    NESTED_OUTER_RESULT_PATH,
    OVERALL_PERFORMANCE_PATH,
    MODEL_COMPARISON_PATH,
    ANTIBIOTIC_PERFORMANCE_PATH,
    SELECTION_FREQUENCY_PATH,
    FULL_COHORT_KERNEL_SELECTION_PATH,
    FULL_COHORT_MODEL_SELECTION_PATH,
    FINAL_SELECTION_SUMMARY_PATH,
    FINAL_MODEL_PATH,
    FINAL_EMBEDDING_PATH,
    FINAL_PATHOGEN_INDEX_PATH,
    FINAL_ANTIBIOTIC_INDEX_PATH,
    FINAL_CONFIGURATION_PATH,
    FINAL_TRAINING_CHECK_PATH,
]

missing_output_files = [
    file_path
    for file_path in files_to_package
    if not file_path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "Notebook 26 output files are missing: "
        f"{missing_output_files}"
    )

if not all(
    outer_fold_checkpoint_is_valid(
        outer_fold_number
    )
    for outer_fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
):
    raise ValueError(
        "At least one nested outer-fold checkpoint failed "
        "final validation."
    )

overall_performance_lookup = overall_performance.set_index(
    "model"
)

model_c_metrics = overall_performance_lookup.loc[
    "Model C"
]

baseline_metrics = overall_performance_lookup.loc[
    "Model 3B baseline"
]

output_manifest = {
    "notebook": 26,
    "model": "Model C",
    "model_c_pathogens": EXPECTED_MODEL_C_PATHOGENS,
    "observed_interactions": MODEL_C_INTERACTIONS,
    "unique_kernel_candidates": len(candidate_catalog),
    "outer_validation_folds": NUMBER_OF_OUTER_FOLDS,
    "inner_selection_folds": NUMBER_OF_INNER_FOLDS,
    "selected_sequence_kernel": FINAL_SEQUENCE_KERNEL,
    "selected_rho": FINAL_RHO,
    "selected_pathogen_dimension": FINAL_PATHOGEN_DIMENSION,
    "selected_ridge_alpha": FINAL_RIDGE_ALPHA,
    "interaction_predictors": FINAL_INTERACTION_DIMENSION,
    "nested_model_c_performance": {
        "mae": float(model_c_metrics["mae"]),
        "rmse": float(model_c_metrics["rmse"]),
        "pearson_r": float(
            model_c_metrics["pearson_r"]
        ),
    },
    "nested_model3b_baseline_performance": {
        "mae": float(baseline_metrics["mae"]),
        "rmse": float(baseline_metrics["rmse"]),
        "pearson_r": float(
            baseline_metrics["pearson_r"]
        ),
    },
    "files": [
        {
            "file_name": file_path.name,
            "size_bytes": file_path.stat().st_size,
            "sha256": file_sha256(file_path),
        }
        for file_path in files_to_package
    ],
    "validation_status": "passed",
}

temporary_manifest_path = Path(
    str(OUTPUT_MANIFEST_PATH)
    + ".partial"
)

with open(
    temporary_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:
    json.dump(
        output_manifest,
        manifest_file,
        indent=2,
    )

temporary_manifest_path.replace(
    OUTPUT_MANIFEST_PATH
)

files_to_package.append(
    OUTPUT_MANIFEST_PATH
)

local_archive_path = (
    WORK_DIRECTORY
    / FINAL_OUTPUT_ARCHIVE_PATH.name
)

local_archive_path.unlink(
    missing_ok=True
)

with zipfile.ZipFile(
    local_archive_path,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=3,
) as archive:
    for file_path in files_to_package:
        archive.write(
            file_path,
            arcname=file_path.name,
        )

with zipfile.ZipFile(
    local_archive_path,
    "r",
) as archive:
    damaged_member = archive.testzip()

    if damaged_member is not None:
        raise ValueError(
            "The local Notebook 26 archive contains a "
            f"damaged file: {damaged_member}"
        )

    archived_members = set(
        archive.namelist()
    )

expected_members = {
    file_path.name
    for file_path in files_to_package
}

if archived_members != expected_members:
    raise ValueError(
        "The Notebook 26 archive member list is incomplete."
    )

local_archive_sha256 = file_sha256(
    local_archive_path
)

partial_archive_path = Path(
    str(FINAL_OUTPUT_ARCHIVE_PATH)
    + ".partial"
)

partial_archive_path.unlink(
    missing_ok=True
)

shutil.copy2(
    local_archive_path,
    partial_archive_path,
)

if file_sha256(
    partial_archive_path
) != local_archive_sha256:
    raise IOError(
        "The copied Notebook 26 archive does not match "
        "the locally validated archive."
    )

partial_archive_path.replace(
    FINAL_OUTPUT_ARCHIVE_PATH
)

with zipfile.ZipFile(
    FINAL_OUTPUT_ARCHIVE_PATH,
    "r",
) as archive:
    if archive.testzip() is not None:
        raise ValueError(
            "The saved Notebook 26 archive failed validation."
        )


final_summary = pd.DataFrame(
    [
        {
            "metric": "Model C pathogens",
            "value": EXPECTED_MODEL_C_PATHOGENS,
        },
        {
            "metric": "Observed MIC interactions",
            "value": MODEL_C_INTERACTIONS,
        },
        {
            "metric": "Unique kernel candidates",
            "value": len(candidate_catalog),
        },
        {
            "metric": "Outer BioSample-grouped folds",
            "value": NUMBER_OF_OUTER_FOLDS,
        },
        {
            "metric": "Inner BioSample-grouped folds",
            "value": NUMBER_OF_INNER_FOLDS,
        },
        {
            "metric": "Nested Model C RMSE",
            "value": float(model_c_metrics["rmse"]),
        },
        {
            "metric": "Nested Model 3B baseline RMSE",
            "value": float(baseline_metrics["rmse"]),
        },
        {
            "metric": "Final sequence kernel",
            "value": FINAL_SEQUENCE_KERNEL,
        },
        {
            "metric": "Final rho",
            "value": FINAL_RHO,
        },
        {
            "metric": "Final pathogen coordinates r_C",
            "value": FINAL_PATHOGEN_DIMENSION,
        },
        {
            "metric": "Final Ridge alpha",
            "value": FINAL_RIDGE_ALPHA,
        },
        {
            "metric": "Final interaction predictors",
            "value": FINAL_INTERACTION_DIMENSION,
        },
        {
            "metric": "Final archive size (MB)",
            "value": round(
                FINAL_OUTPUT_ARCHIVE_PATH.stat().st_size
                / (1024 ** 2),
                3,
            ),
        },
        {
            "metric": "Notebook 26 validation status",
            "value": "passed",
        },
    ]
)

display(final_summary)

print(f"Saved: {OUTPUT_MANIFEST_PATH}")
print(f"Saved: {FINAL_OUTPUT_ARCHIVE_PATH}")

print(
    "\nNotebook 26 completed successfully. Nested "
    "pathogen-out validation and the final Model C "
    "reference model are ready for the next notebook."
)
